# Fully Compressible 2D Euler Solver
## IDC 606 — High Performance Computing

**Scheme:** MUSCL reconstruction + HLLC Riemann solver + TVD-RK3  
**dt:** Constant — computed once from CFL, never changes  
**Snapshots:** t = 0.0, 0.2, 0.4 — each field plotted separately

Run every `%%writefile` cell top-to-bottom, then run the `!python3 main.py` cell.


## 1. Parameters — `para.py`

In [ ]:
%%writefile para.py
import numpy as np

device = "CPU"

# Time
tinit  = 0.0
tfinal = 0.4
dt     = 1e-4         # initial guess; overridden by CFL

# Gas properties
gamma = 1.4
R     = 287.0

# Grid  (used by mesh_and_grid.py — overridden in main.py per case)
Nx = 200
Ny = 50
Lx = 10.0
Ly = 5.0

# Output
t_print    = 0.04
output_dir = "output"


## 2. Mesh and Grid — `mesh_and_grid.py`

In [ ]:
%%writefile mesh_and_grid.py
import numpy as np
import para

Nx = para.Nx
Ny = para.Ny
Lx = para.Lx
Ly = para.Ly

dx = Lx / Nx
dy = Ly / Ny

x_centers = (np.arange(Nx) + 0.5) * dx
y_centers = (np.arange(Ny) + 0.5) * dy

X_mesh, Y_mesh = np.meshgrid(x_centers, y_centers, indexing='ij')

print(f"[mesh] Grid: {Nx} x {Ny}   dx={dx:.4f}  dy={dy:.4f}")


## 3. Compressible State — `compressible.py`

In [ ]:
%%writefile compressible.py
import numpy as np
import mesh_and_grid as grid
import para

gamma = para.gamma

rho = np.ones  ((grid.Nx, grid.Ny))
ux  = np.zeros ((grid.Nx, grid.Ny))
uy  = np.zeros ((grid.Nx, grid.Ny))
p   = np.ones  ((grid.Nx, grid.Ny))

Q = np.zeros((4, grid.Nx, grid.Ny))


def prim_to_cons():
    Q[0] = rho
    Q[1] = rho * ux
    Q[2] = rho * uy
    E    = p / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    Q[3] = E


def cons_to_prim():
    rho[:] = np.maximum(Q[0], 1e-10)
    ux [:]  = Q[1] / rho
    uy [:]  = Q[2] / rho
    e_int   = Q[3] / rho - 0.5 * (ux**2 + uy**2)
    p  [:]  = np.maximum((gamma - 1.0) * rho * e_int, 1e-10)


def flux_x(Qv):
    r  = np.maximum(Qv[0], 1e-10)
    ru = Qv[1];  rv = Qv[2];  E = Qv[3]
    u  = ru / r;  v = rv / r
    e  = E / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    F = np.empty_like(Qv)
    F[0] = ru
    F[1] = ru * u + pp
    F[2] = ru * v
    F[3] = u * (E + pp)
    return F


def flux_y(Qv):
    r  = np.maximum(Qv[0], 1e-10)
    ru = Qv[1];  rv = Qv[2];  E = Qv[3]
    u  = ru / r;  v = rv / r
    e  = E / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    G = np.empty_like(Qv)
    G[0] = rv
    G[1] = rv * u
    G[2] = rv * v + pp
    G[3] = v * (E + pp)
    return G


## 4. MUSCL Reconstruction — `reconstruction.py`

In [ ]:
%%writefile reconstruction.py
import numpy as np


def minmod(a, b):
    return 0.5 * (np.sign(a) + np.sign(b)) * np.minimum(np.abs(a), np.abs(b))


def reconstruct_x(Q):
    k, Nx, Ny = Q.shape
    Qg    = np.concatenate([Q[:, -1:, :], Q, Q[:, :1, :]], axis=1)
    dQ_L  = Qg[:, 1:-1, :] - Qg[:, :-2, :]
    dQ_R  = Qg[:, 2:,   :] - Qg[:, 1:-1, :]
    slope = minmod(dQ_L, dQ_R)
    Q_face_R = Q + 0.5 * slope
    Q_face_L = Q - 0.5 * slope
    Q_L = np.concatenate([Q_face_R[:, -1:, :], Q_face_R], axis=1)
    Q_R = np.concatenate([Q_face_L,             Q_face_L[:, :1, :]], axis=1)
    return Q_L, Q_R


def reconstruct_y(Q):
    k, Nx, Ny = Q.shape
    Qg    = np.concatenate([Q[:, :, -1:], Q, Q[:, :, :1]], axis=2)
    dQ_L  = Qg[:, :, 1:-1] - Qg[:, :, :-2]
    dQ_R  = Qg[:, :, 2:  ] - Qg[:, :, 1:-1]
    slope = minmod(dQ_L, dQ_R)
    Q_face_R = Q + 0.5 * slope
    Q_face_L = Q - 0.5 * slope
    Q_L = np.concatenate([Q_face_R[:, :, -1:], Q_face_R], axis=2)
    Q_R = np.concatenate([Q_face_L,             Q_face_L[:, :, :1]], axis=2)
    return Q_L, Q_R


## 5. HLLC Riemann Solver — `riemann.py`

In [ ]:
%%writefile riemann.py
import numpy as np


def _prim_from_cons(Qv, gamma):
    r  = np.maximum(Qv[0], 1e-10)
    u  = Qv[1] / r
    v  = Qv[2] / r
    e  = Qv[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    a  = np.sqrt(gamma * pp / r)
    return r, u, v, pp, a


def hllc_flux_x(QL, QR, gamma=1.4):
    rL, uL, vL, pL, aL = _prim_from_cons(QL, gamma)
    rR, uR, vR, pR, aR = _prim_from_cons(QR, gamma)

    SL = np.minimum(uL - aL, uR - aR)
    SR = np.maximum(uL + aL, uR + aR)

    num   = pR - pL + rL * uL * (SL - uL) - rR * uR * (SR - uR)
    denom = rL * (SL - uL) - rR * (SR - uR)
    denom = np.where(np.abs(denom) < 1e-14, 1e-14, denom)
    Sstar = num / denom

    EL = QL[3];  ER = QR[3]

    FL = np.empty_like(QL)
    FL[0] = rL * uL;  FL[1] = rL * uL**2 + pL
    FL[2] = rL * uL * vL;  FL[3] = uL * (EL + pL)

    FR = np.empty_like(QR)
    FR[0] = rR * uR;  FR[1] = rR * uR**2 + pR
    FR[2] = rR * uR * vR;  FR[3] = uR * (ER + pR)

    def star_state(Q, r, u, v, p, E, S, Ss):
        coeff = r * (S - u) / (S - Ss)
        Qs    = np.empty_like(Q)
        Qs[0] = coeff
        Qs[1] = coeff * Ss
        Qs[2] = coeff * v
        Qs[3] = coeff * (E / r + (Ss - u) * (Ss + p / (r * (S - u))))
        return Qs

    QL_star = star_state(QL, rL, uL, vL, pL, EL, SL, Sstar)
    QR_star = star_state(QR, rR, uR, vR, pR, ER, SR, Sstar)

    FL_star = FL + SL * (QL_star - QL)
    FR_star = FR + SR * (QR_star - QR)

    F = np.where(SL[np.newaxis, ...]    >= 0, FL,
        np.where(Sstar[np.newaxis, ...] >= 0, FL_star,
        np.where(SR[np.newaxis, ...]    >= 0, FR_star, FR)))
    return F


def hllc_flux_y(QL, QR, gamma=1.4):
    def swap_uv(Qv):
        Qs = Qv.copy()
        Qs[1] = Qv[2]
        Qs[2] = Qv[1]
        return Qs
    F_r = hllc_flux_x(swap_uv(QL), swap_uv(QR), gamma)
    F = F_r.copy()
    F[1] = F_r[2];  F[2] = F_r[1]
    return F


## 6. Flux Divergence — `flux_divergence.py`

In [ ]:
%%writefile flux_divergence.py
import numpy as np
import para
from reconstruction import reconstruct_x, reconstruct_y
from riemann        import hllc_flux_x, hllc_flux_y

gamma = para.gamma


def compute_rhs(Q, dx, dy):
    Nx = Q.shape[1];  Ny = Q.shape[2]

    QL_x, QR_x = reconstruct_x(Q)
    Fx  = hllc_flux_x(QL_x, QR_x, gamma)
    dFx = (Fx[:, 1:Nx+1, :] - Fx[:, 0:Nx, :]) / dx

    QL_y, QR_y = reconstruct_y(Q)
    Fy  = hllc_flux_y(QL_y, QR_y, gamma)
    dFy = (Fy[:, :, 1:Ny+1] - Fy[:, :, 0:Ny]) / dy

    return -(dFx + dFy)


## 7. Time Integration — `fns.py`

In [ ]:
%%writefile fns.py
import numpy as np
import para
from flux_divergence import compute_rhs

gamma = para.gamma


def compute_dt(Q, dx, dy, CFL=0.45):
    r  = np.maximum(Q[0], 1e-10)
    u  = Q[1] / r;  v = Q[2] / r
    e  = Q[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
    a  = np.sqrt(gamma * pp / r)
    sx = max(np.max(np.abs(u) + a), 1e-10)
    sy = max(np.max(np.abs(v) + a), 1e-10)
    return CFL * min(dx / sx, dy / sy)


def time_advance_rk3(Q, dt, dx, dy):
    L0 = compute_rhs(Q,  dx, dy)
    Q1 = Q + dt * L0
    L1 = compute_rhs(Q1, dx, dy)
    Q2 = 0.75 * Q + 0.25 * (Q1 + dt * L1)
    L2 = compute_rhs(Q2, dx, dy)
    return (1.0/3.0) * Q + (2.0/3.0) * (Q2 + dt * L2)


def run_simulation(Q, dx, dy, nsteps,
                   snap_times=(0.0, 0.2, 0.4),
                   print_every=100,
                   CFL=0.45):
    dt = compute_dt(Q, dx, dy, CFL)
    print(f"  Constant dt = {dt:.6e}   (CFL={CFL}, Nsteps={nsteps}, t_final={nsteps*dt:.5f})")
    print()
    print(f"{'Step':>7}  {'t = step*dt':>12}  {'dt':>12}  {'rho_mean':>10}  {'p_mean':>10}")
    print("-" * 62)

    pending   = sorted(s for s in snap_times if s > 0.0)
    snapshots = {}

    if 0.0 in snap_times:
        snapshots[0.0] = Q.copy()
        print("  Snapshot saved at t = 0.000000 (initial state)")

    history = []

    for step in range(1, nsteps + 1):
        Q    = time_advance_rk3(Q, dt, dx, dy)
        Q[0] = np.maximum(Q[0], 1e-10)
        t_now = step * dt

        for st in list(pending):
            if t_now >= st:
                snapshots[st] = Q.copy()
                pending.remove(st)
                print(f"  Snapshot saved at t = {t_now:.6f}  (target t = {st})")

        if step % print_every == 0:
            r  = np.maximum(Q[0], 1e-10)
            u_ = Q[1] / r;  v_ = Q[2] / r
            e  = Q[3] / r - 0.5 * (u_**2 + v_**2)
            pp = np.maximum((gamma - 1.0) * r * e, 1e-10)
            print(f"{step:>7}  {t_now:>12.6f}  {dt:>12.6e}  {r.mean():>10.4f}  {pp.mean():>10.4f}")
            history.append((step, Q.copy()))

        if np.any(~np.isfinite(Q)):
            print(f"  NaN/Inf at step {step} — stopping.")
            break

    print(f"\nDone  steps={nsteps}  t_final={nsteps*dt:.6f}  dt={dt:.6e} (constant)")
    return Q, snapshots, dt, history


## 8. Initial Conditions — `init_fields.py`

In [ ]:
%%writefile init_fields.py
import numpy as np
import para

gamma = para.gamma


def init_sod(Q, X, Y):
    rho = np.where(X < X.max()/2, 1.0,   0.125)
    pp  = np.where(X < X.max()/2, 1.0,   0.1)
    ux  = np.zeros_like(X);  uy = np.zeros_like(X)
    Q[0] = rho;  Q[1] = rho * ux;  Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    print(f"[init] Sod shock tube  — diaphragm at x={X.max()/2:.2f}")
    print(f"       rho range [{rho.min():.3f}, {rho.max():.3f}]   p range [{pp.min():.3f}, {pp.max():.3f}]")


def init_shock_tube_2d(Q, X, Y, rhoL=1.0, pL=1.0, rhoR=0.125, pR=0.1):
    rho = np.where(X < 0.5, rhoL, rhoR)
    pp  = np.where(X < 0.5, pL,   pR)
    ux  = np.zeros_like(X);  uy = np.zeros_like(X)
    Q[0] = rho;  Q[1] = rho * ux;  Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    print(f"[init] 2D Shock Tube — diaphragm at x=0.5")
    print(f"       Left:  rho={rhoL}, p={pL}")
    print(f"       Right: rho={rhoR}, p={pR}")


def init_uniform(Q, X, Y, Mach=2.0, T0=300.0, p0=101325.0, R=287.0):
    a0   = np.sqrt(gamma * R * T0)
    u0   = Mach * a0
    rho0 = p0 / (R * T0)
    rho  = rho0 * np.ones_like(X)
    ux   = u0   * np.ones_like(X)
    uy   = np.zeros_like(X)
    pp   = p0   * np.ones_like(X)
    pp  += 1e-3 * p0 * np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)
    Q[0] = rho;  Q[1] = rho * ux;  Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    print(f"[init] Uniform Mach={Mach:.1f}  rho={rho0:.3f}  p={p0:.1f}")


def init_kh(Q, X, Y, Mach=0.5):
    rho = np.ones_like(X)
    pp  = (1.0 / gamma) * np.ones_like(X)
    a0  = np.sqrt(gamma * pp / rho)
    u0  = Mach * a0
    ux  = np.where(Y < Y.max()/2, u0, -u0)
    np.random.seed(42)
    uy  = 1e-2 * np.random.randn(*X.shape)
    Q[0] = rho;  Q[1] = rho * ux;  Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    print(f"[init] Kelvin-Helmholtz  Mach={Mach:.2f}")


def init_explosion(Q, X, Y, r_blast=0.2, p_in=10.0, p_out=1.0, rho0=1.0):
    xc = X.mean();  yc = Y.mean()
    r2 = (X - xc)**2 + (Y - yc)**2
    rho = rho0 * np.ones_like(X)
    ux  = np.zeros_like(X);  uy = np.zeros_like(X)
    pp  = np.where(r2 < r_blast**2, p_in, p_out)
    Q[0] = rho;  Q[1] = rho * ux;  Q[2] = rho * uy
    Q[3] = pp / (gamma - 1.0) + 0.5 * rho * (ux**2 + uy**2)
    print(f"[init] Circular explosion  r_blast={r_blast}  p_in={p_in}  p_out={p_out}")


## 9. Exact Sod Solution — `sod_exact.py`

In [ ]:
%%writefile sod_exact.py
import numpy as np


def sod_solution(x, t, gamma=1.4, x0=0.5):
    rhoL, uL, pL = 1.0,   0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1
    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)

    def fL(p):
        if p > pL:
            A = 2.0 / ((gamma+1)*rhoL);  B = (gamma-1)/(gamma+1)*pL
            return (p - pL) * np.sqrt(A / (p + B))
        return (2*aL/(gamma-1)) * ((p/pL)**((gamma-1)/(2*gamma)) - 1)

    def fR(p):
        if p > pR:
            A = 2.0 / ((gamma+1)*rhoR);  B = (gamma-1)/(gamma+1)*pR
            return (p - pR) * np.sqrt(A / (p + B))
        return (2*aR/(gamma-1)) * ((p/pR)**((gamma-1)/(2*gamma)) - 1)

    def f(p):   return fL(p) + fR(p) + (uR - uL)
    def df(p):
        dp = 1e-6 * p
        return (f(p+dp) - f(p-dp)) / (2*dp)

    p_star = 0.5 * (pL + pR)
    for _ in range(100):
        dp = -f(p_star) / df(p_star);  p_star += dp
        if abs(dp) < 1e-12 * p_star:  break

    u_star    = 0.5*(uL+uR) + 0.5*(fR(p_star) - fL(p_star))
    aL_star   = aL * (p_star/pL)**((gamma-1)/(2*gamma))
    S_HL      = uL - aL
    S_TL      = u_star - aL_star
    S_contact = u_star
    S_R       = uR + aR * np.sqrt((gamma+1)/(2*gamma)*(p_star/pR) + (gamma-1)/(2*gamma))
    rhoL_star = rhoL * (p_star/pL)**(1/gamma)
    rhoR_star = rhoR * ((p_star/pR + (gamma-1)/(gamma+1)) / ((gamma-1)/(gamma+1)*p_star/pR + 1))

    if t <= 0:
        return (np.where(x < x0, rhoL, rhoR),
                np.where(x < x0, uL,   uR),
                np.where(x < x0, pL,   pR))

    xi  = (x - x0) / t
    rho = np.empty_like(x, dtype=float)
    u_f = np.empty_like(x, dtype=float)
    pp  = np.empty_like(x, dtype=float)

    for i, s in enumerate(xi):
        if   s <= S_HL:
            rho[i]=rhoL;  u_f[i]=uL;  pp[i]=pL
        elif s <= S_TL:
            u_tmp  = 2/(gamma+1)*(aL + (gamma-1)/2*uL + s)
            a_tmp  = aL + (gamma-1)/2*(uL - u_tmp)
            rho[i] = rhoL*(a_tmp/aL)**(2/(gamma-1))
            u_f[i] = u_tmp
            pp[i]  = pL*(a_tmp/aL)**(2*gamma/(gamma-1))
        elif s <= S_contact:
            rho[i]=rhoL_star;  u_f[i]=u_star;  pp[i]=p_star
        elif s <= S_R:
            rho[i]=rhoR_star;  u_f[i]=u_star;  pp[i]=p_star
        else:
            rho[i]=rhoR;  u_f[i]=uR;  pp[i]=pR

    return rho, u_f, pp


## 10. Plotting — `saving.py`

In [ ]:
%%writefile saving.py
import numpy as np
import matplotlib.pyplot as plt
import os, para

gamma = para.gamma
os.makedirs("output", exist_ok=True)

_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']

_FIELD_META = {
    'rho'      : ("Density  rho",     "rho",   "viridis"),
    'ux'       : ("Velocity  u",      "u",     "RdBu_r"),
    'p'        : ("Pressure  p",      "p",     "plasma"),
    'mach'     : ("Mach Number  M",   "M",     "hot"),
    'schlieren': ("Schlieren |grad rho|", "|grad rho|", "gray"),
}


def _prim(Q):
    r  = np.maximum(Q[0], 1e-10)
    u  = Q[1] / r;  v = Q[2] / r
    ei = Q[3] / r - 0.5 * (u**2 + v**2)
    pp = np.maximum((gamma - 1.0) * r * ei, 1e-10)
    a  = np.sqrt(gamma * pp / r)
    M  = np.sqrt(u**2 + v**2) / a
    return r, u, v, pp, a, M

def _sorted_snaps(snapshots):
    return sorted(snapshots.items(), key=lambda x: x[0])

def _extract_1d(Q, field):
    r, u, v, pp, a, M = _prim(Q)
    return {'rho': r, 'ux': u, 'p': pp, 'mach': M}[field].mean(axis=1)

def _extract_2d(Q, field, dx, dy):
    r, u, v, pp, a, M = _prim(Q)
    if field == 'rho':    return r
    if field == 'ux':     return u
    if field == 'p':      return pp
    if field == 'mach':   return M
    if field == 'schlieren':
        gx = np.gradient(r, dx, axis=0)
        gy = np.gradient(r, dy, axis=1)
        gm = np.sqrt(gx**2 + gy**2)
        return np.exp(-10.0 * gm / (gm.max() + 1e-14))
    raise ValueError(field)


def plot_sod_field(snapshots, X, field='rho', save=True):
    x1d = X[:, 0]
    long, short, _ = _FIELD_META[field]
    snaps = _sorted_snaps(snapshots)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.set_title(f"Sod Shock Tube — {long}  (t = 0 / 0.2 / 0.4)", fontsize=12)
    ax.set_xlabel("x");  ax.set_ylabel(short)
    for (t_val, Q), col in zip(snaps, _COLORS):
        ax.plot(x1d, _extract_1d(Q, field), color=col, lw=2.0, label=f"t = {t_val:.2f}")
    ax.legend(fontsize=9);  ax.grid(True, alpha=0.3);  plt.tight_layout()
    if save:
        fname = f"output/sod_{field}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved -> {fname}")
    plt.show()


def plot_sod_verification(snapshots, X, save=True):
    from sod_exact import sod_solution
    x1d   = X[:, 0]
    snaps = _sorted_snaps(snapshots)
    fig, axes = plt.subplots(3, 3, figsize=(14, 10))
    fig.suptitle("Sod Shock Tube — Exact vs MUSCL-HLLC  (constant dt)", fontsize=13)
    fkeys = ['rho', 'ux', 'p'];  ylabs = ["Density rho", "Velocity u", "Pressure p"]
    for ci, (t_val, Q) in enumerate(snaps):
        exact = sod_solution(x1d, t_val)
        for ri, (fk, yl) in enumerate(zip(fkeys, ylabs)):
            ax = axes[ri, ci]
            ax.plot(x1d, exact[ri],          'k-',  lw=2.0, label="Exact")
            ax.plot(x1d, _extract_1d(Q, fk), '--',  color=_COLORS[ci], lw=1.5, label="MUSCL-HLLC")
            ax.set_title(f"t = {t_val:.2f}", fontsize=10)
            if ci == 0: ax.set_ylabel(yl, fontsize=9)
            ax.set_xlabel("x", fontsize=9);  ax.legend(fontsize=7);  ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save:
        fname = "output/sod_verification_all.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved -> {fname}")
    plt.show()


def plot_2d_field(snapshots, X, Y, dx, dy, field='rho', tag='sim', save=True):
    snaps = _sorted_snaps(snapshots)
    long, short, cmap = _FIELD_META[field]
    all_data = [_extract_2d(Q, field, dx, dy) for _, Q in snaps]
    vmin = min(d.min() for d in all_data)
    vmax = max(d.max() for d in all_data)
    if vmin == vmax: vmax = vmin + 1e-6
    n = len(snaps)
    fig, axes = plt.subplots(1, n, figsize=(5.5*n, 5), squeeze=False)
    fig.suptitle(f"2-D Simulation — {long}", fontsize=13)
    for ci, ((t_val, Q), data) in enumerate(zip(snaps, all_data)):
        ax = axes[0, ci]
        if field == 'schlieren':
            im = ax.imshow(data.T, origin='lower',
                           extent=[X.min(), X.max(), Y.min(), Y.max()],
                           cmap=cmap, vmin=0, vmax=1, aspect='auto')
        else:
            im = ax.contourf(X, Y, data, levels=40, cmap=cmap, vmin=vmin, vmax=vmax)
            if field == 'mach':
                ax.contour(X, Y, data, levels=[1.0], colors='white', linewidths=1.2, linestyles='--')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=short)
        ax.set_title(f"t = {t_val:.2f}", fontsize=11)
        ax.set_xlabel("x");  ax.set_ylabel("y");  ax.set_aspect('equal')
    plt.tight_layout()
    if save:
        fname = f"output/{tag}_{field}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved -> {fname}")
    plt.show()


def plot_energy_history(history, dx, dy, save=True):
    steps  = [h[0]                    for h in history]
    E_tots = [h[1][3].sum() * dx * dy for h in history]
    E0     = E_tots[0] if E_tots[0] != 0 else 1.0
    E_rel  = [(e - E_tots[0]) / abs(E0) * 100 for e in E_tots]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle("Energy Conservation  (constant dt)", fontsize=12)
    axes[0].plot(steps, E_tots, 'b-o', ms=4)
    axes[0].set_xlabel("Step");  axes[0].set_ylabel("Total Energy")
    axes[0].set_title("Absolute Total Energy");  axes[0].grid(True, alpha=0.3)
    axes[1].plot(steps, E_rel, 'r-o', ms=4)
    axes[1].set_xlabel("Step");  axes[1].set_ylabel("Relative Error (%)")
    axes[1].set_title("Energy Conservation Error");  axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    if save:
        fname = "output/energy_conservation.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved -> {fname}")
    plt.show()


def save_all_sod(snapshots, X, Y, dx, dy, history):
    print("\n-- Sod: individual field plots --")
    for field in ('rho', 'ux', 'p', 'mach'):
        plot_sod_field(snapshots, X, field=field)
    plot_sod_verification(snapshots, X)
    plot_energy_history(history, dx, dy)


def save_all_2d(snapshots, X, Y, dx, dy, history, tag='sim'):
    print(f"\n-- 2-D [{tag}]: individual field plots --")
    for field in ('rho', 'ux', 'p', 'mach', 'schlieren'):
        plot_2d_field(snapshots, X, Y, dx, dy, field=field, tag=tag)
    plot_energy_history(history, dx, dy)


## 11. Main Driver — `main.py`

In [ ]:
%%writefile main.py
import numpy as np
import time
import os

os.makedirs("output", exist_ok=True)

import para
import mesh_and_grid as grid
from init_fields import init_sod, init_explosion, init_shock_tube_2d
from fns         import run_simulation
from saving      import save_all_sod, save_all_2d

SNAP_TIMES = [0.0, 0.2, 0.4]


def run_sod():
    print("\n" + "="*60)
    print("  CASE 1 - Sod Shock Tube  (constant dt)")
    print("="*60)
    Nx, Ny = 200, 50
    Lx, Ly = 10.0, 5.0
    dx, dy = Lx/Nx, Ly/Ny
    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')
    Q = np.zeros((4, Nx, Ny))
    init_sod(Q, X, Y)
    t0 = time.time()
    Q_final, snapshots, dt, history = run_simulation(
        Q, dx, dy, nsteps=400, snap_times=SNAP_TIMES, print_every=100, CFL=0.45)
    print(f"  Wall time: {time.time()-t0:.2f} s")
    print(f"  Snapshots: {sorted(snapshots.keys())}")
    save_all_sod(snapshots, X, Y, dx, dy, history)
    return Q_final, snapshots, dt, history


def run_explosion():
    print("\n" + "="*60)
    print("  CASE 2 - 2-D Circular Explosion  (constant dt)")
    print("="*60)
    Nx, Ny = 200, 200
    Lx, Ly = 1.0, 1.0
    dx, dy = Lx/Nx, Ly/Ny
    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')
    Q = np.zeros((4, Nx, Ny))
    init_explosion(Q, X, Y, r_blast=0.2, p_in=10.0, p_out=1.0)
    t0 = time.time()
    Q_final, snapshots, dt, history = run_simulation(
        Q, dx, dy, nsteps=800, snap_times=SNAP_TIMES, print_every=100, CFL=0.40)
    print(f"  Wall time: {time.time()-t0:.2f} s")
    print(f"  Snapshots: {sorted(snapshots.keys())}")
    save_all_2d(snapshots, X, Y, dx, dy, history, tag='explosion')
    return Q_final, snapshots, dt, history


def run_shock_tube_2d():
    print("\n" + "="*60)
    print("  CASE 3 - 2D Shock Tube (left/right pressure split)")
    print("="*60)
    Nx, Ny = 200, 200
    Lx, Ly = 1.0, 1.0
    dx, dy = Lx/Nx, Ly/Ny
    x = (np.arange(Nx) + 0.5) * dx
    y = (np.arange(Ny) + 0.5) * dy
    X, Y = np.meshgrid(x, y, indexing='ij')
    Q = np.zeros((4, Nx, Ny))
    init_shock_tube_2d(Q, X, Y)
    t0 = time.time()
    Q_final, snapshots, dt, history = run_simulation(
        Q, dx, dy, nsteps=400, snap_times=SNAP_TIMES, print_every=100, CFL=0.45)
    print(f"  Wall time: {time.time()-t0:.2f} s")
    print(f"  Snapshots: {sorted(snapshots.keys())}")
    save_all_2d(snapshots, X, Y, dx, dy, history, tag='shock2d')
    return Q_final, snapshots, dt, history


if __name__ == "__main__":
    run_sod()
    run_explosion()
    run_shock_tube_2d()
    print("\n  All cases complete.  Results saved to ./output/")


## 12. Run All Cases

In [ ]:
!python3 main.py

## 13. Display All Output Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

for f in sorted(os.listdir('output')):
    if f.endswith('.png'):
        img = mpimg.imread(f'output/{f}')
        plt.figure(figsize=(12, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f.replace('.png','').replace('_',' '))
        plt.tight_layout()
        plt.show()
